In [ ]:
# Path configuration
from pathlib import Path

PROJECT_ROOT = Path("..")  # Repository root.
BASELINE = PROJECT_ROOT / "data"
FIRM_DATA = PROJECT_ROOT
PATSTAT = PROJECT_ROOT / "patstat"
PATENT_TEXT = PROJECT_ROOT / "patent_text"
SAMPLEDATA = PROJECT_ROOT / "sampledata"
DATA = Path("data")          # Generated revision intermediates.
OUTPUTS = Path("outputs")    # Tables, figures and saved comparison results.
SEED = 42


# Prepare alliance comparisons and corporate-group screening

Run from `revision_notebooks/` after `../7.2_complement_firm_ctry.ipynb` has saved `complement_reg.parquet` and `firm_complement.parquet`. Set their directory with `FIRM_DATA` in the first cell. Both `supply` (candidate organization) and `demand` (focal firm) hold PSN IDs.

Exclude pairs of a firm with itself. Calculate Fig. 4a/b statistics, select each firm's most complementary candidate, and identify co-assignments between companies in different countries for corporate-group screening. Save the pairs and classifications under `DATA/alliance/`, and figure statistics under `OUTPUTS/alliance/`. Notebooks 12 and 13 read these files.


In [ ]:
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd


regression = FIRM_DATA / 'complement_reg.parquet'
candidates = FIRM_DATA / 'firm_complement.parquet'
intermediate = DATA / 'alliance'
intermediate.mkdir(parents=True, exist_ok=True)
output = OUTPUTS / 'alliance'
output.mkdir(parents=True, exist_ok=True)

LABELS = ['[1-2]', '[3-5]', '[6-10]', '[11-50]', '[51-100]', '>100']

## Read inputs and calculate Fig. 4a

Exclude pairs of a firm with itself. Plot mean complementarity with intervals of 1.96 times the standard error.


In [ ]:
def moments(count, total, squares):
    count, total, squares = (np.asarray(x) for x in (count, total, squares))
    mean = total / count
    std = np.sqrt((squares - total * mean) / (count - 1))
    sem = std / np.sqrt(count)
    return pd.DataFrame(dict(count=count, total=total, mean=mean,
                             std=std, sem=sem, ci95=1.96 * sem))

In [ ]:
con = duckdb.connect()
con.read_parquet(str(regression)).create_view('reg')
# Keep the stored row order for exact ties in the two ranking variables.
con.read_parquet(str(candidates), file_row_number=True).create_view('candidates')
a = con.sql('''SELECT copatgreen, count(*) AS n, sum(n_gbb) AS total,
    sum(n_gbb * n_gbb) AS squares FROM reg WHERE supply != demand
    GROUP BY copatgreen ORDER BY copatgreen''').df()
a_stats = moments(a.n, a.total, a.squares)
a_stats.insert(0, 'copatgreen', a.copatgreen.astype(bool))
a_stats.to_csv(output / 'fig4a_statistics.csv', index=False)

## Select co-assignments between companies in different countries

Keep pairs of distinct organizations that share a green patent and whose types are both `COMPANY`. Convert countries to uppercase, remove surrounding spaces, and require different two-letter country codes, excluding XX/ZZ/WO/EP. Count each pair once, with the lower PSN ID as Applicant A.


In [ ]:
con.execute('''CREATE TEMP TABLE observed AS SELECT supply, demand FROM reg
    WHERE supply != demand AND copatgreen > 0''')
# Source table already includes PATSTAT names, sector labels and R&D countries.
# Canonical orientation is lower ID as Applicant A; deduplicate both directions.
observed = con.sql('''SELECT DISTINCT least(c.supply,c.demand) AS psn_id_low,
    greatest(c.supply,c.demand) AS psn_id_high,
    CASE WHEN c.supply<c.demand THEN supplyfirm ELSE demandfirm END AS name_low,
    CASE WHEN c.supply<c.demand THEN demandfirm ELSE supplyfirm END AS name_high,
    CASE WHEN c.supply<c.demand THEN upper(trim(supplyctry)) ELSE upper(trim(demandctry)) END AS country_low,
    CASE WHEN c.supply<c.demand THEN upper(trim(demandctry)) ELSE upper(trim(supplyctry)) END AS country_high,
    supplytype AS type_low, demandtype AS type_high
    FROM candidates c JOIN observed o USING(supply,demand)
    WHERE supplytype = 'COMPANY' AND demandtype = 'COMPANY'
    AND regexp_full_match(upper(trim(supplyctry)), '[A-Z]{2}')
    AND regexp_full_match(upper(trim(demandctry)), '[A-Z]{2}')
    AND upper(trim(supplyctry)) NOT IN ('XX','ZZ','WO','EP')
    AND upper(trim(demandctry)) NOT IN ('XX','ZZ','WO','EP')
    AND upper(trim(supplyctry)) != upper(trim(demandctry))
    ORDER BY psn_id_low, psn_id_high''').df()
observed.to_csv(intermediate / 'observed_company_pairs.csv', index=False)

## Rank candidate organizations and calculate Fig. 4b

Exclude the focal firm from its own candidates. Sort by decreasing `n_gbb`, then decreasing `log_n_pat`, preserving table order for exact ties. Rank candidates within each focal firm across the batches read from disk. Remove surrounding spaces from country values and use nonblank entries for the country comparison.


In [ ]:
ids = con.sql('SELECT DISTINCT demand FROM candidates ORDER BY demand').df().demand.to_numpy()
counters = np.zeros(len(ids), dtype=np.int64)
counts, successes = np.zeros(6, dtype=np.int64), np.zeros(6, dtype=np.int64)
top = []
reader = con.execute('''SELECT c.* EXCLUDE(file_row_number),
    (o.supply IS NOT NULL) AS copatgreen
    FROM candidates c LEFT JOIN observed o USING(supply,demand)
    WHERE c.supply != c.demand
    ORDER BY n_gbb DESC, log_n_pat DESC, file_row_number''').fetch_record_batch(500_000)
for batch in reader:
    frame = batch.to_pandas()
    idx = np.searchsorted(ids, frame.demand.to_numpy())
    rank = pd.Series(idx).groupby(idx, sort=False).cumcount().to_numpy() + 1 + counters[idx]
    counters += np.bincount(idx, minlength=len(ids))
    bins = np.searchsorted([2, 5, 10, 50, 100], rank, side='left')
    counts += np.bincount(bins, minlength=6)
    successes += np.bincount(bins, weights=frame.copatgreen, minlength=6).astype(np.int64)
    top.append(frame.loc[rank == 1].drop(columns='copatgreen'))
top = pd.concat(top).sort_values('demand')
for column in ['supplyctry', 'demandctry']:
    top[column] = top[column].str.strip()
top['country_observed'] = top.supplyctry.ne('') & top.demandctry.ne('')
top['cross_border'] = top.country_observed & top.supplyctry.ne(top.demandctry)
top.to_parquet(intermediate / 'top_candidates.parquet', index=False)
b_stats = moments(counts, successes, successes)
b_stats.insert(0, 'rnkgroup', LABELS)
b_stats.to_csv(output / 'fig4b_statistics.csv', index=False)
con.close()

top.head()

## Corporate-group screening prompts

Use the prompts below to classify whether two organizations belong to the same corporate group. Separate prompts cover observed co-assignments and selected candidates. Use names and general knowledge, with an uncertain category when ownership changes over 1976–2022 prevent a clear classification.


In [ ]:
SYSTEM_PROMPT = """You are evaluating organizational identity for a patent-assignee robustness audit.

For each pair, decide whether the two PATSTAT standardized applicants belong to the same ultimate corporate group.

Classification rules:
- same_group: clear alternative names for the same company, a parent and controlled subsidiary, two controlled subsidiaries, or a regional legal entity and its corporate parent.
- different_group: clearly distinct, independently controlled organizations. Shared country, industry, or generic name words are not evidence of common ownership.
- uncertain: ownership cannot be determined reliably from the supplied names and well-established general knowledge, including relationships that may be time-specific.
- The patent portfolios span 1976--2022 and no pair-specific date is supplied. Use uncertain rather than guessing when an acquisition, divestiture, or historical relationship could change the answer.
- Organization country describes the main country of R&D activity in this study, not necessarily legal incorporation.
- A company and an unrelated university, public institute, hospital, or government body are different_group.
- Do not infer a relationship from the numerical identifiers.

Return only JSON with these fields:
- classification: same_group, different_group, or uncertain
- confidence: integer 1--5
- shared_group: ultimate group name if classification is same_group, otherwise an empty string
- evidence_basis: name_equivalence, known_ownership, clearly_distinct, or insufficient_information
- rationale: one concise sentence grounded in the organization names or a well-established ownership relationship."""

def top_candidate_prompt(task):
    return f"""Applicant A (focal firm)
Standardized name: {task['demand_name']}
Organization type: {task['demand_type']}
R&D country: {task['demand_country'] or '[not recorded]'}

Applicant B (highest-complementarity candidate)
Standardized name: {task['supply_name']}
Organization type: {task['supply_type']}
R&D country: {task['supply_country'] or '[not recorded]'}

Do Applicant A and Applicant B belong to the same ultimate corporate group? Return the required JSON object only."""

def observed_pair_prompt(task):
    return f"""Applicant A
Standardized name: {task['demand_name']}
Organization type: {task['demand_type']}
R&D country: {task['demand_country'] or '[not recorded]'}

Applicant B
Standardized name: {task['supply_name']}
Organization type: {task['supply_type']}
R&D country: {task['supply_country'] or '[not recorded]'}

Do Applicant A and Applicant B belong to the same ultimate corporate group? Return the required JSON object only."""

## Generate and save the corporate-group classifications

Set `OPENAI_API_KEY`. The final cell screens co-assignment pairs and selected firm–candidate pairs using DeepSeek-V4-Flash with temperature 0, reasoning effort high and JSON output. Save raw responses and classification CSVs under `DATA/alliance/`.

`save_judgments(...)` joins the classifications to the corresponding pairs and saves the tables read by notebooks 12 and 13.


In [ ]:
def tasks_from_candidates(frame):
    return [dict(task_id=f'd{r.demand}_s{r.supply}',
        demand_psn_id=int(r.demand), demand_name=r.demandfirm, demand_type=r.demandtype,
        demand_country=r.demandctry.strip(), supply_psn_id=int(r.supply),
        supply_name=r.supplyfirm, supply_type=r.supplytype, supply_country=r.supplyctry.strip())
        for r in frame.itertuples()]

def tasks_from_observed(frame):
    return [dict(task_id=f'd{r.psn_id_low}_s{r.psn_id_high}',
        demand_psn_id=int(r.psn_id_low), demand_name=r.name_low, demand_type=r.type_low,
        demand_country=r.country_low, supply_psn_id=int(r.psn_id_high),
        supply_name=r.name_high, supply_type=r.type_high, supply_country=r.country_high)
        for r in frame.itertuples()]

In [ ]:
def save_judgments(tasks, cohort, response_path):
    """Join saved responses to their task IDs and write the downstream CSV."""
    response_path = Path(response_path)
    with response_path.open(encoding='utf-8') as handle:
        responses = [json.loads(line) for line in handle]
    judgments = pd.DataFrame([
        dict(task_id=row['task_id'], returned_model=row['returned_model'], **row['judgment'])
        for row in responses])
    result = pd.DataFrame(tasks).merge(judgments, on='task_id')
    if cohort == 'observed_pairs':
        result['psn_id_low'] = result.demand_psn_id
        result['psn_id_high'] = result.supply_psn_id
        filename = 'observed_judgments.csv'
    else:
        filename = 'top_candidate_judgments.csv'
    result.to_csv(response_path.parent / filename, index=False)
    return result


def run(tasks, cohort, response_path):
    """Use OPENAI_API_KEY from the environment; save raw and normalized results."""
    from openai import OpenAI
    client = OpenAI(base_url='https://api.deepseek.com')
    prompt = {'top_candidates': top_candidate_prompt, 'observed_pairs': observed_pair_prompt}[cohort]
    with Path(response_path).open('w', encoding='utf-8') as handle:
        for task in tasks:
            response = client.chat.completions.create(model='deepseek-v4-flash',
                temperature=0, reasoning_effort='high', response_format={'type':'json_object'},
                messages=[{'role':'system','content':SYSTEM_PROMPT},
                          {'role':'user','content':prompt(task)}])
            row = dict(task_id=task['task_id'], judgment=json.loads(response.choices[0].message.content),
                       returned_model=response.model)
            handle.write(json.dumps(row, ensure_ascii=False)+'\n')
    return save_judgments(tasks, cohort, response_path)


In [ ]:
observed_judgments = run(tasks_from_observed(observed), 'observed_pairs',
                         intermediate / 'observed_responses.jsonl')
top_judgments = run(tasks_from_candidates(top), 'top_candidates',
                    intermediate / 'top_candidate_responses.jsonl')
